# PERF-CANDIDATE. Constraint to Candidate and Cost Transfer | 从约束到候选与代价转移

**难度：** Hard | **环境：** CPU 机制练习 + 单卡 GPU 机制实验  
**标签：** `资源约束` `候选契约` `代价转移` `证伪条件` | **目标人群：** 性能优化与系统决策学习者

---

## 本节导读

性能候选必须作用于已经确认的约束。减少 FLOPs、搬运字节或峰值容量是不同方向，新增计算、通信、转换和质量风险也不能相互抵消成一个任意分数。

本节从资源约束出发，建立候选效果向量与硬门槛，再输出进入真实测量、继续调整或提前拒绝的候选状态。


## 前置阅读

- [Part 01 · 02 参数量与 FLOPs](../01_Hardware_Math_and_Systems/02_LLM_Params_and_FLOPs.ipynb)：建立计算工作量账本。
- [Part 01 · 03 GPU 架构与显存](../01_Hardware_Math_and_Systems/03_GPU_Architecture_and_Memory.ipynb)：区分计算、带宽与容量约束。
- [Part 02 · PERF-ROOFLINE](./PERF-ROOFLINE_Arithmetic_Intensity_and_Resource_Ceiling.ipynb)：取得资源上界证据。
---


### Step 1：候选必须命中已确认约束

资源约束由工作负载和证据共同定义。计算约束对应浮点运算或计算路径，带宽约束对应最低搬运字节，容量约束对应峰值时仍驻留的对象。候选若只改变非主约束，即使局部指标变化，也不能据此预期端到端收益。

| 约束 | 候选应直接改变的量 | 进入测量前需要的证据 |
| --- | --- | --- |
| 计算 | FLOPs、计算形状或计算内核效率 | 算术强度、经验计算上界与目标路径 |
| 带宽 | 搬运字节、数据复用或中间状态读写 | 最低字节账本、经验带宽与目标路径 |
| 容量 | 跨越峰值的对象大小或生命周期 | 峰值驻留对象、释放时机与失败边界 |
| 关键路径 | 暴露等待、同步依赖或串行阶段 | 事件依赖、重叠区间与暴露时间 |

![从已确认约束到候选契约](../docs/public/02_PyTorch_Algorithms/perf_candidate_contract.svg)


### Step 2：减少量与新增代价必须分轴记录

候选可能同时减少一个资源量并增加另一个资源量。例如低位宽表示减少搬运字节和驻留容量，却可能增加格式转换、反量化计算和质量风险。各变化保留原始单位与方向，不能用任意权重合并。

| 变化轴 | 负值表示 | 正值表示 |
| --- | --- | --- |
| 计算量 | 运算减少 | 新增重算或转换运算 |
| 搬运字节 | 数据读写减少 | 新增读写或中间状态 |
| 峰值容量 | 峰值占用下降 | 新增常驻或临时空间 |
| 暴露时间 | 关键路径缩短 | 新增等待或同步 |
| 质量变化 | 质量改善 | 质量损失 |

![候选效果向量与硬门槛](../docs/public/02_PyTorch_Algorithms/perf_candidate_cost_vector.svg)


### Step 3：硬门槛与证伪条件决定是否进入测量

作用方向正确只说明候选值得继续检查。正确性、质量、容量和执行兼容性属于硬门槛；目标减少量不足、目标路径未命中或新增代价超过预算属于证伪条件。

| 状态 | 条件 | 动作 |
| --- | --- | --- |
| 进入测量 | 命中主约束且硬门槛可满足 | 进入真实基线—候选对照 |
| 继续调整 | 方向正确，但减少量不足或新增代价过高 | 调整粒度、范围或执行路径 |
| 提前拒绝 | 未命中主约束或硬门槛失败 | 停止该候选，不用其他收益抵消失败 |


### Step 4：实现候选效果向量与准入判断

题目区接收已确认约束、候选变化量和硬门槛预算。先验证变化轴，再检查候选是否减少主约束，最后按硬门槛与新增代价输出状态。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1 / `normalize_effects` | 保留五个资源轴及其正负方向 | 不接受未知轴、缺失轴或非数值 | 完整向量与非法输入 |
| TODO 2 / `primary_reduction` | 找到约束对应轴并读取减少量 | 主约束必须严格下降 | 命中与未命中约束 |
| TODO 3 / `decide_candidate` | 比较质量与新增代价预算 | 不使用加权总分抵消失败 | `measure / tune / reject` 三种状态 |


In [ ]:
RESOURCE_AXES = ("compute_flops", "memory_bytes", "peak_bytes", "exposed_ms", "quality_loss")
CONSTRAINT_AXIS = {"compute": "compute_flops", "bandwidth": "memory_bytes", "capacity": "peak_bytes", "critical_path": "exposed_ms"}


In [ ]:
def normalize_effects(effects: dict[str, float]) -> dict[str, float]:
    """校验并返回完整的候选效果向量。"""
    # ==========================================
    # TODO 1：规范效果向量
    # ==========================================
    # unknown_axes = ???  # 不属于 RESOURCE_AXES 的字段集合
    # missing_axes = ???  # RESOURCE_AXES 中尚未提供的字段集合
    # normalized = ???  # 转为 float 后的完整效果向量
    raise NotImplementedError("TODO 1：请规范效果向量")
    if unknown_axes or missing_axes:
        raise ValueError(f"效果轴不匹配：unknown={sorted(unknown_axes)}, missing={sorted(missing_axes)}")
    if any(not isinstance(value, (int, float)) for value in effects.values()):
        raise TypeError("效果值必须是数值")
    return normalized


def primary_reduction(constraint: str, effects: dict[str, float]) -> float:
    """返回主约束轴的减少量；正数表示约束得到缓解。"""
    if constraint not in CONSTRAINT_AXIS:
        raise ValueError("constraint 必须是 compute、bandwidth、capacity 或 critical_path")
    # ==========================================
    # TODO 2：检查主约束减少量
    # ==========================================
    # target_axis = ???  # 当前约束对应的效果轴
    # reduction = ???  # 负向变化转换为正的减少量
    raise NotImplementedError("TODO 2：请计算主约束减少量")
    return reduction


def decide_candidate(constraint: str, effects: dict[str, float], budgets: dict[str, float]) -> dict[str, object]:
    """按主约束、质量门槛和新增代价预算判断候选状态。"""
    normalized = normalize_effects(effects)
    reduction = primary_reduction(constraint, normalized)
    required = {"min_primary_reduction", "max_quality_loss", "max_added_compute", "max_added_memory", "max_added_peak", "max_added_exposed_ms"}
    if set(budgets) != required:
        raise ValueError("budgets 字段不完整")
    # ==========================================
    # TODO 3：执行硬门槛决策
    # ==========================================
    # quality_passed = ???  # 质量损失是否未越过门槛
    # added_costs_passed = ???  # 四类新增代价是否均在预算内
    # primary_missed = ???  # 主约束没有得到任何缓解
    raise NotImplementedError("TODO 3：请完成候选决策")
    if primary_missed or not quality_passed:
        decision = "reject"
    elif reduction < budgets["min_primary_reduction"] or not added_costs_passed:
        decision = "tune"
    else:
        decision = "measure"
    return {"decision": decision, "primary_reduction": reduction, "quality_passed": quality_passed, "added_costs_passed": added_costs_passed}


### 测试

运行下方测试，检查变化轴完整性、主约束方向和三类候选状态。


In [ ]:
def _budgets():
    """返回测试使用的硬门槛预算。"""
    return {"min_primary_reduction": 10.0, "max_quality_loss": 0.02, "max_added_compute": 20.0, "max_added_memory": 5.0, "max_added_peak": 5.0, "max_added_exposed_ms": 1.0}


def test_effect_vector_and_constraint_direction():
    """效果向量必须完整，主约束减少量使用正确的资源轴。"""
    effects = {"compute_flops": 5, "memory_bytes": -30, "peak_bytes": -20, "exposed_ms": 0.2, "quality_loss": 0.01}
    assert normalize_effects(effects)["memory_bytes"] == -30.0
    assert primary_reduction("bandwidth", effects) == 30.0
    assert primary_reduction("critical_path", effects) == -0.2
    try:
        normalize_effects({**effects, "score": 1})
    except ValueError:
        pass
    else:
        raise AssertionError("未知效果轴必须报错")


def test_measure_tune_and_reject():
    """方向、预算和质量门槛分别产生三类决策。"""
    measure = {"compute_flops": 5, "memory_bytes": -30, "peak_bytes": -20, "exposed_ms": 0.2, "quality_loss": 0.01}
    tune = {**measure, "memory_bytes": -5}
    miss = {**measure, "memory_bytes": 2}
    reject = {**measure, "quality_loss": 0.05}
    assert decide_candidate("bandwidth", measure, _budgets())["decision"] == "measure"
    assert decide_candidate("bandwidth", tune, _budgets())["decision"] == "tune"
    assert decide_candidate("bandwidth", miss, _budgets())["decision"] == "reject"
    assert decide_candidate("bandwidth", reject, _budgets())["decision"] == "reject"


test_effect_vector_and_constraint_direction()
test_measure_tune_and_reject()
print("All tests passed!")


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
RESOURCE_AXES = ("compute_flops", "memory_bytes", "peak_bytes", "exposed_ms", "quality_loss")
CONSTRAINT_AXIS = {"compute": "compute_flops", "bandwidth": "memory_bytes", "capacity": "peak_bytes", "critical_path": "exposed_ms"}

def normalize_effects(effects: dict[str, float]) -> dict[str, float]:
    """校验并返回完整的候选效果向量。"""
    # TODO 1：规范效果向量。
    unknown_axes = set(effects) - set(RESOURCE_AXES)
    missing_axes = set(RESOURCE_AXES) - set(effects)
    if unknown_axes or missing_axes:
        raise ValueError(f"效果轴不匹配：unknown={sorted(unknown_axes)}, missing={sorted(missing_axes)}")
    if any(not isinstance(value, (int, float)) for value in effects.values()):
        raise TypeError("效果值必须是数值")
    return {axis: float(effects[axis]) for axis in RESOURCE_AXES}


def primary_reduction(constraint: str, effects: dict[str, float]) -> float:
    """返回主约束轴的减少量；正数表示约束得到缓解。"""
    if constraint not in CONSTRAINT_AXIS:
        raise ValueError("constraint 必须是 compute、bandwidth、capacity 或 critical_path")
    # TODO 2：检查主约束减少量。
    target_axis = CONSTRAINT_AXIS[constraint]
    reduction = -normalize_effects(effects)[target_axis]
    return reduction


def decide_candidate(constraint: str, effects: dict[str, float], budgets: dict[str, float]) -> dict[str, object]:
    """按主约束、质量门槛和新增代价预算判断候选状态。"""
    normalized = normalize_effects(effects)
    reduction = primary_reduction(constraint, normalized)
    required = {"min_primary_reduction", "max_quality_loss", "max_added_compute", "max_added_memory", "max_added_peak", "max_added_exposed_ms"}
    if set(budgets) != required:
        raise ValueError("budgets 字段不完整")
    # TODO 3：执行硬门槛决策。
    quality_passed = normalized["quality_loss"] <= budgets["max_quality_loss"]
    added_costs_passed = (max(0.0, normalized["compute_flops"]) <= budgets["max_added_compute"] and max(0.0, normalized["memory_bytes"]) <= budgets["max_added_memory"] and max(0.0, normalized["peak_bytes"]) <= budgets["max_added_peak"] and max(0.0, normalized["exposed_ms"]) <= budgets["max_added_exposed_ms"])
    primary_missed = reduction <= 0
    if primary_missed or not quality_passed:
        decision = "reject"
    elif reduction < budgets["min_primary_reduction"] or not added_costs_passed:
        decision = "tune"
    else:
        decision = "measure"
    return {"decision": decision, "primary_reduction": reduction, "quality_passed": quality_passed, "added_costs_passed": added_costs_passed}


### 解析

#### TODO 1：规范效果向量

- 五个效果轴始终存在，避免候选通过省略不利维度获得错误结论。
- 测试加入未知字段，确认实现不会接受无法解释的综合分数。

#### TODO 2：检查主约束减少量

- 变化向量使用负数表示减少，因此主约束缓解量取对应轴的相反数。
- 约束到效果轴的映射是显式契约，不从候选名称推断。

#### TODO 3：执行硬门槛决策

- 质量失败或主约束没有下降时直接拒绝；方向正确但减少量不足或新增代价过高时继续调整。
- 测试分别构造可测量、减少量不足、方向错误和质量失败候选，覆盖三条决策路径。


### Step 5：GPU 机制实验——约束到单候选实测矩阵

CPU 题目确认候选作用方向、代价轴和硬门槛；GPU 实验在同一解码器形状 MLP 流水线上比较 BF16 表示与双缓冲重叠两个单项候选，检查它们是否改变预期对象并保持输出质量。

| 学习问题 | CPU 已确认 | GPU 要确认与后续使用 |
| --- | --- | --- |
| 候选是否真正命中声明的约束 | 效果向量、主约束方向与候选状态 | BF16 的模型字节与执行时间变化；重叠路径的暴露时间变化；逐负载质量、延迟、峰值与阶段服务时间 |

#### 5.1 环境与固定工作负载

本实验使用 `PERF-CUDA-MECHANISM`。首次运行按[性能分析实验 Quickstart](../topic_discussion/performance_analysis/quickstart.md#perf-cuda-mechanism-quickstart)创建并预检环境，实验身份与结果路径登记在[性能优化实验资产表](../topic_discussion/performance_optimization/experiment_assets.md)。本页使用固定合成张量，不下载模型或数据集。

| 配置层 | 本节默认口径 | 修改规则 |
| --- | --- | --- |
| 运行环境 | 单张 CUDA GPU；`PERF-CUDA-MECHANISM` | Profile 预检未返回 `status=ok` 时停止 |
| 工作负载 | SwiGLU MLP；隐藏维度 2048；中间维度 5504；8 个流水线批次 | 基线和候选共享权重、输入、批次数与随机种子 |
| 负载矩阵 | 每批 32、256 Token | 改变形状、候选或测量协议后使用新运行标签 |
| 单项候选 | BF16 串行；FP32 双缓冲重叠 | 本页不运行 BF16+重叠组合，组合归 PERF-COMPOSITION |
| 测量协议 | 12 轮轮换预热；24 轮逐轮轮换三条路径的执行顺序 | 每轮保留各路径样本，不按整段先后顺序测量 |
| 决策门槛 | 余弦相似度不低于 0.999；中位数至少改善 3%；P95 不得回归超过 5% | 质量失败直接拒绝；收益不足或尾延迟回归记为调整 |
| 结果资产 | `benchmarks/results/perf_candidate/<semantic_id>_<run_label>.json` | 正式复测使用唯一标签 |


In [3]:
RUN_MODE = "real_gpu"  # 默认值 dry_run；改为 real_gpu 执行真实候选矩阵。
RUN_LABEL = "rtx5070ti_candidate_rotating_v2"  # 默认复测标签；更换硬件或协议后改为新的唯一标签。
PROFILE_ID = "PERF-CUDA-MECHANISM"  # 默认性能机制环境。
SEMANTIC_ID = "PERF-CANDIDATE"  # 稳定机制 ID。
WORKLOAD_ID = "perf-decoder-mlp-candidate-matrix"  # 稳定工作负载 ID。
WORKLOAD_VERSION = "v2"  # v2 使用逐轮轮换顺序，减少路径测量的先后偏差。
GPU_CONFIG = {
    "hidden_size": 2048,          # 默认值 2048；SwiGLU 输入与输出维度。
    "intermediate_size": 5504,   # 默认值 5504；SwiGLU 中间维度。
    "token_counts": [32, 256],   # 默认两档 Token 负载；候选必须逐负载判断。
    "pipeline_batches": 8,       # 默认值 8；双缓冲路径的连续批次数。
    "warmup_steps": 12,          # 默认值 12；三条路径轮换预热且不计入统计。
    "repeat_steps": 24,          # 默认值 24；每轮轮换路径顺序并统计中位数与 P95。
    "minimum_speedup": 1.03,     # 默认值 1.03；采用候选要求的最低中位数加速比。
    "minimum_cosine": 0.999,     # 默认值 0.999；BF16 输出质量门槛。
    "maximum_p95_ratio": 1.05,   # 默认值 1.05；候选 P95 最多允许回归 5%。
    "seed": 42,                  # 默认值 42；固定权重与输入。
}
RESULTS_RELATIVE_DIR = "benchmarks/results/perf_candidate"


#### 5.2 执行单候选矩阵并保存 JSON

下方单元运行 FP32 串行基线、BF16 串行候选和 FP32 双缓冲候选。每条路径保存首次执行、稳态中位数/P95、模型字节、峰值显存、搬运与计算服务时间以及输出余弦相似度。


In [4]:
import json, platform
import torch
from tools.performance_optimization_runtime import run_candidate_composition_matrix
from tools.project_runtime import resolve_project_root

RESULTS_DIR = resolve_project_root() / RESULTS_RELATIVE_DIR
if RUN_MODE == "real_gpu":
    matrix = run_candidate_composition_matrix(
        hidden_size=GPU_CONFIG["hidden_size"], intermediate_size=GPU_CONFIG["intermediate_size"],
        token_counts=GPU_CONFIG["token_counts"], pipeline_batches=GPU_CONFIG["pipeline_batches"],
        warmup_steps=GPU_CONFIG["warmup_steps"], repeat_steps=GPU_CONFIG["repeat_steps"],
        seed=GPU_CONFIG["seed"],
        variants=["baseline_fp32_serial", "bf16_serial", "fp32_overlap"],
    )
    if matrix.get("protocol", {}).get("measurement_order") != "rotating_order_within_each_round":
        raise RuntimeError("当前内核仍缓存旧版性能运行时；请重启内核后从 5.1 开始重新运行。")
    loads = {name: {item["token_count"]: item for item in data["loads"]} for name, data in matrix["variants"].items()}
    comparisons = []
    for candidate in ("bf16_serial", "fp32_overlap"):
        for token_count in GPU_CONFIG["token_counts"]:
            base, current = loads["baseline_fp32_serial"][token_count], loads[candidate][token_count]
            speedup = base["latency"]["median_ms"] / current["latency"]["median_ms"]
            p95_ratio = current["latency"]["p95_ms"] / base["latency"]["p95_ms"]
            quality_passed = current["cosine_vs_fp32"] >= GPU_CONFIG["minimum_cosine"]
            tail_passed = p95_ratio <= GPU_CONFIG["maximum_p95_ratio"]
            direction_passed = (current["model_bytes"] < base["model_bytes"] if candidate == "bf16_serial" else current["latency"]["median_ms"] < base["latency"]["median_ms"])
            if not quality_passed or not direction_passed:
                decision = "reject"
            elif speedup >= GPU_CONFIG["minimum_speedup"] and tail_passed:
                decision = "accept_for_load"
            else:
                decision = "tune"
            comparisons.append({
                "candidate": candidate, "token_count": token_count, "speedup": speedup,
                "p95_ratio": p95_ratio,
                "model_bytes_reduction": 1.0 - current["model_bytes"] / base["model_bytes"],
                "active_bytes_reduction": 1.0 - (current["model_bytes"] + current["buffer_bytes"]) / (base["model_bytes"] + base["buffer_bytes"]),
                "matrix_peak_allocated_bytes": current["matrix_peak_allocated_bytes"],
                "cosine_vs_fp32": current["cosine_vs_fp32"],
                "direction_passed": direction_passed, "quality_passed": quality_passed, "tail_passed": tail_passed,
                "decision": decision,
            })
    record = {
        "schema_version": "performance-candidate-mechanism/v1", "profile_id": PROFILE_ID,
        "semantic_id": SEMANTIC_ID, "run_label": RUN_LABEL,
        "framework": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda},
        "hardware": {"device": torch.cuda.get_device_name(0), "total_memory_bytes": torch.cuda.get_device_properties(0).total_memory},
        "workload": {"id": WORKLOAD_ID, "version": WORKLOAD_VERSION, **GPU_CONFIG},
        "matrix": matrix, "comparisons": comparisons, "failure": None,
        "evidence_level": "local_mechanism_benchmark",
        "decision": "review_per_load",
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    result_path = RESULTS_DIR / f"{SEMANTIC_ID.lower()}_{RUN_LABEL}.json"
    result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"result_path": str(result_path), "comparisons": comparisons}, ensure_ascii=False, indent=2))
else:
    print({"run_mode": RUN_MODE, "profile_id": PROFILE_ID, "semantic_id": SEMANTIC_ID, "config": GPU_CONFIG})


{
  "result_path": "/data/llm-algo-leetcode-datawhale/llm-algo-leetcode/benchmarks/results/perf_candidate/perf-candidate_rtx5070ti_candidate_rotating_v2.json",
  "comparisons": [
    {
      "candidate": "bf16_serial",
      "token_count": 32,
      "speedup": 2.6507768209725073,
      "p95_ratio": 0.4887795525932068,
      "model_bytes_reduction": 0.5,
      "active_bytes_reduction": 0.5,
      "matrix_peak_allocated_bytes": 378668032,
      "cosine_vs_fp32": 0.9999672770500183,
      "direction_passed": true,
      "quality_passed": true,
      "tail_passed": true,
      "decision": "accept_for_load"
    },
    {
      "candidate": "bf16_serial",
      "token_count": 256,
      "speedup": 2.875548744911817,
      "p95_ratio": 0.3300163641603307,
      "model_bytes_reduction": 0.5,
      "active_bytes_reduction": 0.5,
      "matrix_peak_allocated_bytes": 547489792,
      "cosine_vs_fp32": 1.0,
      "direction_passed": true,
      "quality_passed": true,
      "tail_passed": true,
   

#### 5.3 读取结果

读取同一 `RUN_LABEL` 的结果文件，先检查失败与质量门槛，再逐负载比较候选方向、稳态加速、模型字节和峰值变化。


In [5]:
if RUN_MODE == "real_gpu":
    result_path = RESULTS_DIR / f"{SEMANTIC_ID.lower()}_{RUN_LABEL}.json"
    if not result_path.is_file():
        raise FileNotFoundError(f"未找到结果，请先运行 5.2：{result_path}")
    candidate_record = json.loads(result_path.read_text(encoding="utf-8"))
    print(json.dumps(candidate_record["comparisons"], ensure_ascii=False, indent=2))
else:
    candidate_record = {"status": "not_run"}
    print(candidate_record)


[
  {
    "candidate": "bf16_serial",
    "token_count": 32,
    "speedup": 2.6507768209725073,
    "p95_ratio": 0.4887795525932068,
    "model_bytes_reduction": 0.5,
    "active_bytes_reduction": 0.5,
    "matrix_peak_allocated_bytes": 378668032,
    "cosine_vs_fp32": 0.9999672770500183,
    "direction_passed": true,
    "quality_passed": true,
    "tail_passed": true,
    "decision": "accept_for_load"
  },
  {
    "candidate": "bf16_serial",
    "token_count": 256,
    "speedup": 2.875548744911817,
    "p95_ratio": 0.3300163641603307,
    "model_bytes_reduction": 0.5,
    "active_bytes_reduction": 0.5,
    "matrix_peak_allocated_bytes": 547489792,
    "cosine_vs_fp32": 1.0,
    "direction_passed": true,
    "quality_passed": true,
    "tail_passed": true,
    "decision": "accept_for_load"
  },
  {
    "candidate": "fp32_overlap",
    "token_count": 32,
    "speedup": 1.0228878140362858,
    "p95_ratio": 0.9768108721139892,
    "model_bytes_reduction": 0.0,
    "active_bytes_reduction

#### 5.4 解释证据与决策

RTX 5070 Ti 的 v2 实测采用 24 轮配对样本，并在每轮轮换三条路径的执行顺序。BF16 在两档负载均通过质量、方向和尾延迟门槛；FP32 重叠只在 256 Token 达到最低收益门槛。

| 观察项 | 读取字段 | v2 实测结果 | 解释与动作 |
| --- | --- | --- | --- |
| 运行有效性 | `failure`、`matrix.protocol` | 无失败；24 轮配对测量并逐轮轮换顺序 | 结果可用于比较当前单卡、形状和候选实现 |
| BF16 质量与容量 | 余弦相似度、模型与活动字节 | 余弦相似度不低于 0.999；字节减少 50% | 正确性与容量方向同时通过 |
| BF16 延迟 | 中位数加速比、P95 比值 | 32/256 Token 约 2.65×/2.88×；P95 比值约 0.49/0.33 | 两档负载均接受 BF16 单项 |
| FP32 重叠 | 中位数加速比、P95 比值 | 32 Token 约 1.02×；256 Token 约 1.03×；P95 均未回归 | 小负载继续调优；只对 256 Token 接受 |
| 后续出口 | 逐负载决策 | BF16 是稳定单项；重叠收益依赖负载 | 进入 PERF-COMPOSITION 检查组合边际，并保留 BF16 回退路径 |


## 相关阅读

- [性能优化 · 02 资源约束与候选代价](../topic_discussion/performance_optimization/02_observability_and_profiling.md)
- [PERF-ROOFLINE：算术强度与资源上界](./PERF-ROOFLINE_Arithmetic_Intensity_and_Resource_Ceiling.ipynb)
- [PERF-REGRESSION：配对基准与性能回归阈值](./PERF-REGRESSION_Paired_Benchmark_and_Regression_Thresholds.ipynb)
